In [3]:
!pip install chromadb sentence-transformers -q

In [2]:
!pip install sentence-transformers chromadb groq pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.4 MB/s eta 0:00:00


In [4]:
# =========================================================
# STEP 1: IMPORT LIBRARIES
# =========================================================

import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer

from groq import Groq

In [5]:
# =========================================================
# STEP 2: LOAD DATASET
# =========================================================

print("Loading Dataset....")

df = pd.read_csv("/content/drive/MyDrive/15 days intern dataset/college_notes.csv")

print("Dataset Loaded Successfully")

print("Total Notes:", len(df))

print("Subjects:")
print(df['subject'].value_counts())

Loading Dataset....
Dataset Loaded Successfully
Total Notes: 15
Subjects:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64


In [6]:
# =========================================================
# STEP 3: PREPARE DOCUMENTS
# =========================================================

documents = df['content'].tolist()

ids = [
    f"note_{row['note_id']}"
    for row in df.to_dict('records')
]

metadatas = [

    {
        "subject": row['subject'],
        "topic": row['topic']
    }

    for row in df.to_dict('records')
]

print("\nDocuments Prepared:", len(documents))


Documents Prepared: 15


In [7]:
# =========================================================
# STEP 4: LOAD EMBEDDING MODEL
# =========================================================

print("\nLoading Embedding Model....")

embedding_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

print("Embedding Model Loaded Successfully")


Loading Embedding Model....


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded Successfully


In [8]:
# =========================================================
# STEP 5: GENERATE EMBEDDINGS
# =========================================================

print("\nGenerating Embeddings....")

embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
)

print("Embedding Shape:", embeddings.shape)


Generating Embeddings....


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding Shape: (15, 384)


In [9]:
# =========================================================
# STEP 6: SETUP CHROMADB
# =========================================================

print("\nCreating ChromaDB Client....")

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="college_notes_rag"
)

print("Collection Created Successfully")


Creating ChromaDB Client....
Collection Created Successfully


In [10]:
# =========================================================
# STEP 7: STORE DOCUMENTS IN CHROMADB
# =========================================================

collection.add(

    documents=documents,

    embeddings=embeddings.tolist(),

    ids=ids,

    metadatas=metadatas
)

print("\nDocuments Indexed Successfully")

print("Total Documents:",
      collection.count())


Documents Indexed Successfully
Total Documents: 15


In [11]:
# =========================================================
# STEP 8: CREATE RETRIEVAL FUNCTION
# =========================================================

def retrieve_relevant_chunks(question, top_k=3):

    # Convert question into embedding
    question_embedding = embedding_model.encode(
        question
    ).tolist()

    # Search similar notes
    results = collection.query(

        query_embeddings=[question_embedding],

        n_results=top_k
    )

    return results

In [12]:
# =========================================================
# STEP 9: BUILD CONTEXT
# =========================================================

def build_context_from_results(results):

    context_parts = []

    for i, (doc, meta) in enumerate(

        zip(
            results['documents'][0],
            results['metadatas'][0]
        )
    ):

        context_text = f"""
Result {i+1}

Subject: {meta['subject']}

Topic: {meta['topic']}

Content:
{doc}
"""

        context_parts.append(context_text)

    final_context = "\n".join(context_parts)

    return final_context

In [13]:
# =========================================================
# STEP 10: SETUP GROQ CLIENT
# =========================================================

# Replace with your Groq API key

client = Groq(
    api_key="YOUR_GROQ_API_KEY"
)

print("\nGroq Client Connected")




Groq Client Connected


In [26]:
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer

from groq import Groq

# =========================================================
# STEP 11: COMPLETE RAG PIPELINE
# =========================================================

def ask_college_assistant(
        question,
        top_k=3,
        verbose=True
):

    # =========================================
    # RETRIEVE RELEVANT NOTES
    # =========================================

    if verbose:

        print("\nQuestion:", question)

        print("-" * 50)

        print(
            "Retrieving Relevant Notes...."
        )

    results = retrieve_relevant_chunks(
        question,
        top_k=top_k
    )

    # =========================================
    # BUILD CONTEXT
    # =========================================

    context = build_context_from_results(
        results
    )

    if verbose:

        print("\nRetrieved Context:")

        print("-" * 50)

        print(context)

    # =========================================
    # SYSTEM PROMPT
    # =========================================

    system_prompt = """
You are a helpful academic assistant
for engineering students.

Use only the provided context to answer.

Rules:
1. Give accurate answers.
2. Keep explanations beginner-friendly.
3. If answer is unavailable in context,
   say:
   'I could not find relevant information.'
4. Do not hallucinate.
"""

    # =========================================
    # USER PROMPT
    # =========================================

    user_prompt = f"""
Context:
{context}

Question:
{question}

Answer the question using only
the provided context.
"""

    # =========================================
    # GENERATE LLM RESPONSE
    # =========================================

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    # =========================================
    # PRINT FINAL ANSWER
    # =========================================

    if verbose:

        print("\nFinal AI Answer:")

        print("-" * 50)

        print(answer)

    return answer


In [27]:
from groq import Groq

# Add your Groq API Key here
client = Groq(
    api_key="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"
)

print("Groq Client Connected Successfully")

Groq Client Connected Successfully


In [28]:
# =========================================================
# STEP 12: TEST THE SYSTEM
# =========================================================

question = """
What is ETL and why is it important
in data engineering?
"""

answer = ask_college_assistant(question)


Question: 
What is ETL and why is it important
in data engineering?

--------------------------------------------------
Retrieving Relevant Notes....

Retrieved Context:
--------------------------------------------------

Result 1

Subject: Data Engineering

Topic: ETL Pipelines

Content:
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.


Result 2

Subject: Data Engineering

Topic: APIs and Data Collection

Content:
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.


Result 3

Subject: Python Programming

Topic: Data Visualization

Content:
Data visualization is the process of representing data as charts graphs and visual formats. Python libraries 